In [ ]:
from typing import TypedDict,Literal
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thingking":{
            "type":"disabled"
        }
    }
)

#1. 状態を定義
class OverAllState(TypedDict):
    topic: str
    poem: str
    joke: str
    content_type: str

#2. ノードを定義
def node_a(state: OverAllState) -> OverAllState:
    poem = model.invoke([f"{state['topic']}をテーマにした詩を書いてください"]).content
    return {
        "poem": poem
    }

def node_b(state: OverAllState) -> OverAllState:
    joke = model.invoke([f"{state['topic']}をテーマにしたジョークを書いてください"]).content
    return {
        "joke": joke
    }

def my_route(state: OverAllState) -> Literal["node_a","node_b"]:
    if "詩" in state["content_type"]:
        return "node_a"
    else:
        return "node_b"


#3. グラフを構築
builder = StateGraph(state_schema=OverAllState)
builder.add_node(node_a)
builder.add_node(node_b)
builder.add_conditional_edges(START,my_route)
builder.add_edge("node_b",END)
builder.add_edge("node_a",END)

graph = builder.compile()
poem_res = graph.invoke({"topic": "猫","content_type":"詩"})
print(poem_res)

joke_res = graph.invoke({"topic": "猫","content_type":"ジョーク"})
print(joke_res)

from IPython.display import display
display(graph)
